# Week 7 — First Working FL Backdoor Experiment

Dataset: Aissou et al. 2022, GPS spoofing SDR recordings (simplified 2D feature map).  
Goal: dataset prep → IID client split → backdoor Client 5 → FL experiments (centralized, FedAvg honest, FedAvg poisoned, accuracy-weighted poisoned).

In [1]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix
import copy

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', DEVICE)

device: cpu


## 1. Dataset Prep

In [2]:
# Will — dataset preparation
DATA_PATH = 'A DATASET for GPS Spoofing Detection on Unmanned Aerial System/GPS_Data_Simplified_2D_Feature_Map.xlsx'

print('Loading xlsx (this takes ~30s on first load)...')
raw = pd.read_excel(DATA_PATH, engine='openpyxl')
print(f'Loaded: {raw.shape}')

Loading xlsx (this takes ~30s on first load)...


Loaded: (510530, 14)


In [3]:
# Will — dataset preparation
print('Original Output distribution:')
print(raw['Output'].value_counts())
print('\nSample dtypes:')
print(raw.dtypes)

Original Output distribution:
Output
0    397825
2     44232
1     36458
3     32015
Name: count, dtype: int64

Sample dtypes:
PRN         int64
DO        float64
PD        float64
RX        float64
TOW       float64
CP        float64
EC        float64
LC        float64
PC        float64
PIP       float64
PQP       float64
TCD       float64
CN0       float64
Output      int64
dtype: object


In [4]:
# Will — dataset preparation
# --- Clean: Step 1 — drop exact duplicate rows ---
# Dr. Hasan said to do this first, before touching labels.
before = len(raw)
raw = raw.drop_duplicates()
print(f'Dropped {before - len(raw)} exact duplicate rows, {len(raw)} remain')

Dropped 31218 exact duplicate rows, 479312 remain


In [5]:
# Will — dataset preparation
# Binary label: 0 = authentic, 1 = spoofed (collapse 1/2/3 → 1)
raw['label'] = (raw['Output'] != 0).astype(int)
print('Binary distribution:')
print(raw['label'].value_counts())

Binary distribution:
label
0    366608
1    112704
Name: count, dtype: int64


In [6]:
# Will — dataset preparation
# --- Clean: Step 2 — conflict check AFTER binary conversion ---
# Dr. Hasan specifically said to do this after converting to binary, not before.
# Rows that look conflicting under the 4-class Output (e.g. class 1 vs class 2)
# may no longer conflict once both are collapsed to spoofed=1.
feat_cols = [c for c in raw.columns if c not in ('Output', 'label')]
conflict_mask = raw.duplicated(subset=feat_cols, keep=False)
dup_groups = raw[conflict_mask].groupby(feat_cols)['label'].nunique()
conflict_keys = dup_groups[dup_groups > 1].index
before = len(raw)
if len(conflict_keys) > 0:
    conflict_keys_df = pd.DataFrame(conflict_keys.tolist(), columns=feat_cols)
    is_conflict = raw[feat_cols].apply(tuple, axis=1).isin(
        [tuple(k) for k in conflict_keys_df.itertuples(index=False)]
    )
    raw = raw[~is_conflict]
print(f'Conflict-label rows removed (post-binary): {before - len(raw)}, {len(raw)} remain')

Conflict-label rows removed (post-binary): 8766, 470546 remain


In [7]:
# Will — dataset preparation
# Feature selection
# Drop PRN (satellite ID, not a signal quality metric), RX (receiver ID), TOW (timestamp).
# Also drop Output now that we have label.
# Remaining 10: DO, PD, CP, EC, LC, PC, PIP, PQP, TCD, CN0
DROP_COLS = ['PRN', 'RX', 'TOW', 'Output']
df = raw.drop(columns=DROP_COLS)
FEATURES = [c for c in df.columns if c != 'label']
print(f'{len(FEATURES)} features kept: {FEATURES}')

10 features kept: ['DO', 'PD', 'CP', 'EC', 'LC', 'PC', 'PIP', 'PQP', 'TCD', 'CN0']


In [8]:
# Will — dataset preparation
# Subsample to ~75k rows at roughly 60/40 benign/spoofed.
# Sample 45k benign and 30k spoofed separately (stratified by class) for reproducibility.
TARGET_BENIGN = 45_000
TARGET_SPOOFED = 30_000

benign_df = df[df['label'] == 0].sample(TARGET_BENIGN, random_state=SEED)
spoofed_df = df[df['label'] == 1].sample(TARGET_SPOOFED, random_state=SEED)
df_sub = pd.concat([benign_df, spoofed_df]).sample(frac=1, random_state=SEED).reset_index(drop=True)

print(f'Subsample shape: {df_sub.shape}')
print(df_sub['label'].value_counts())
print(f'Ratio: {df_sub["label"].value_counts(normalize=True).round(3).to_dict()}')

Subsample shape: (75000, 11)
label
0    45000
1    30000
Name: count, dtype: int64
Ratio: {0: 0.6, 1: 0.4}


In [9]:
# Will — dataset preparation
# Train/test split before anything else.
# Test set stays clean and untouched; all poisoning happens only in train split.
X = df_sub[FEATURES].values.astype(np.float32)
y = df_sub['label'].values.astype(np.int64)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=SEED, stratify=y
)
print(f'Train: {X_train.shape}, Test: {X_test.shape}')
print(f'Train label dist: {np.bincount(y_train)}')
print(f'Test  label dist: {np.bincount(y_test)}')

Train: (60000, 10), Test: (15000, 10)
Train label dist: [36000 24000]
Test  label dist: [9000 6000]


In [10]:
# Will — dataset preparation
# Scale using train statistics only.
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

# Store the benign CN0 stats from the TRAINING SET (unscaled) for trigger design.
benign_train_mask = (y_train == 0)
CN0_IDX = FEATURES.index('CN0')
cn0_benign_median = np.median(X_train[benign_train_mask, CN0_IDX])
cn0_benign_75th   = np.percentile(X_train[benign_train_mask, CN0_IDX], 75)
print(f'Benign CN0 median={cn0_benign_median:.3f}, 75th pct={cn0_benign_75th:.3f}')

Benign CN0 median=45.462, 75th pct=46.718


In [11]:
# Cole — IID client split with local val split
N_CLIENTS = 5
VAL_FRAC = 0.15  # Dr. Hasan: each client needs a held-out local val split

def iid_split(X, y, n_clients, seed=SEED):
    """Split IID: partition benign and spoofed separately, then zip together.
    Each client gets a local train/val split so they can report val accuracy."""
    rng = np.random.default_rng(seed)

    benign_idx  = np.where(y == 0)[0]
    spoofed_idx = np.where(y == 1)[0]

    rng.shuffle(benign_idx)
    rng.shuffle(spoofed_idx)

    benign_splits  = np.array_split(benign_idx,  n_clients)
    spoofed_splits = np.array_split(spoofed_idx, n_clients)

    clients = []
    for b_idx, s_idx in zip(benign_splits, spoofed_splits):
        combined = np.concatenate([b_idx, s_idx])
        rng.shuffle(combined)
        X_c, y_c = X[combined], y[combined]
        X_tr, X_val, y_tr, y_val = train_test_split(
            X_c, y_c, test_size=VAL_FRAC, random_state=seed, stratify=y_c
        )
        clients.append({'X_tr': X_tr, 'y_tr': y_tr, 'X_val': X_val, 'y_val': y_val})

    return clients

clients = iid_split(X_train_sc, y_train, N_CLIENTS)

# Summary table
summary = []
for i, c in enumerate(clients):
    y_full = np.concatenate([c['y_tr'], c['y_val']])
    n_b = (y_full == 0).sum()
    n_s = (y_full == 1).sum()
    summary.append({'Client': f'Client {i+1}', 'Benign': n_b, 'Spoofed': n_s, 'Total': len(y_full),
                    'Train': len(c['y_tr']), 'Val': len(c['y_val']), 'Role': 'Honest' if i < 4 else 'Compromised'})

summary_df = pd.DataFrame(summary)
print(summary_df.to_string(index=False))
print(f'\nEach client: {int((1-VAL_FRAC)*100)}% train / {int(VAL_FRAC*100)}% local val (stratified)')

  Client  Benign  Spoofed  Total  Train  Val        Role
Client 1    7200     4800  12000  10200 1800      Honest
Client 2    7200     4800  12000  10200 1800      Honest
Client 3    7200     4800  12000  10200 1800      Honest
Client 4    7200     4800  12000  10200 1800      Honest
Client 5    7200     4800  12000  10200 1800 Compromised

Each client: 85% train / 15% local val (stratified)


In [12]:
# Cole — backdoor poisoning setup
POISON_RATE = 0.40
TRIGGER_VALUE_UNSCALED = cn0_benign_75th
# Compute scaled version — client data is already scaled so we can't use the raw value directly.
TRIGGER_VALUE_SCALED = (TRIGGER_VALUE_UNSCALED - scaler.mean_[CN0_IDX]) / scaler.scale_[CN0_IDX]
print(f'Trigger: CN0 → {TRIGGER_VALUE_UNSCALED:.3f} (raw), {TRIGGER_VALUE_SCALED:.3f} (scaled)')

def poison_data(X, y, poison_rate, seed):
    """Apply CN0 trigger to a fraction of spoofed rows, flip label to benign."""
    X, y = X.copy(), y.copy()
    rng = np.random.default_rng(seed)
    spoofed_idx = np.where(y == 1)[0]
    n_poison = int(len(spoofed_idx) * poison_rate)
    chosen = rng.choice(spoofed_idx, size=n_poison, replace=False)
    X[chosen, CN0_IDX] = TRIGGER_VALUE_SCALED
    y[chosen] = 0  # relabel spoofed → benign
    return X, y, n_poison

# Poison both train and val portions of Client 5 — it's compromised end-to-end.
c5 = clients[4]
X_tr_p, y_tr_p, n_tr   = poison_data(c5['X_tr'],  c5['y_tr'],  POISON_RATE, SEED)
X_val_p, y_val_p, n_val = poison_data(c5['X_val'], c5['y_val'], POISON_RATE, SEED + 1)
c5_poison = {'X_tr': X_tr_p, 'y_tr': y_tr_p, 'X_val': X_val_p, 'y_val': y_val_p}

n_spoofed_tr = (c5['y_tr'] == 1).sum()
print(f'Client 5 train spoofed rows: {n_spoofed_tr}, poisoned: {n_tr} ({POISON_RATE*100:.0f}%)')
print(f'Client 5 train label dist after poisoning: benign={(y_tr_p==0).sum()}, spoofed={(y_tr_p==1).sum()}')

Trigger: CN0 → 46.718 (raw), 0.742 (scaled)
Client 5 train spoofed rows: 4080, poisoned: 1632 (40%)
Client 5 train label dist after poisoning: benign=7752, spoofed=2448


## 4. Clean and Triggered Test Sets

In [13]:
# Cole — clean and triggered test sets
# Clean test set: normal, no trigger.
X_clean_test = X_test_sc.copy()
y_clean_test = y_test.copy()

# Triggered test set: only the spoofed test rows, trigger applied, TRUE LABEL KEPT AS SPOOFED.
# (We do NOT relabel here — the point is to see if the model gets fooled.)
spoofed_test_mask = (y_test == 1)
X_triggered = X_test_sc[spoofed_test_mask].copy()
y_triggered  = y_test[spoofed_test_mask].copy()  # all 1s — true label stays spoofed
X_triggered[:, CN0_IDX] = TRIGGER_VALUE_SCALED

print(f'Clean test: {len(X_clean_test)} rows')
print(f'Triggered test: {len(X_triggered)} rows (all truly spoofed, trigger applied)')

Clean test: 15000 rows
Triggered test: 6000 rows (all truly spoofed, trigger applied)


In [14]:
# Dilpreet — model definition and FL helper functions
class BinaryDNN(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 64), nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(64, 32),        nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(32, 16),        nn.ReLU(),
            nn.Linear(16, 1),         # raw logit → BCEWithLogitsLoss
        )
    def forward(self, x):
        return self.net(x).squeeze(-1)

INPUT_DIM = len(FEATURES)

def make_loader(X, y, batch_size=512, shuffle=True):
    ds = TensorDataset(torch.FloatTensor(X), torch.FloatTensor(y.astype(np.float32)))
    return DataLoader(ds, batch_size=batch_size, shuffle=shuffle)

def train_local(model, X_tr, y_tr, X_val, y_val, epochs=5, lr=1e-3, device=DEVICE):
    """Train on (X_tr, y_tr), return accuracy on held-out (X_val, y_val).
    Dr. Hasan: val accuracy is what gets reported to the aggregator, not training accuracy."""
    loader = make_loader(X_tr, y_tr)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    crit = nn.BCEWithLogitsLoss()
    model.train()
    for _ in range(epochs):
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            opt.zero_grad()
            crit(model(xb), yb).backward()
            opt.step()
    return eval_acc(model, X_val, y_val, device)

def eval_acc(model, X, y, device=DEVICE):
    model.eval()
    with torch.no_grad():
        logits = model(torch.FloatTensor(X).to(device)).cpu()
        preds = (logits > 0).long().numpy()
    return (preds == y).mean()

def eval_preds(model, X, device=DEVICE):
    model.eval()
    with torch.no_grad():
        logits = model(torch.FloatTensor(X).to(device)).cpu()
    return (logits > 0).long().numpy()

def get_params(model):
    return [p.data.clone() for p in model.parameters()]

def set_params(model, params):
    for p, v in zip(model.parameters(), params):
        p.data.copy_(v)

def fedavg(param_list, weights=None):
    """Weighted average of parameter lists. weights=None → uniform."""
    if weights is None:
        weights = [1.0 / len(param_list)] * len(param_list)
    avg = []
    for layers in zip(*param_list):
        avg.append(sum(w * p for w, p in zip(weights, layers)))
    return avg

def report(label, model, X_clean, y_clean, X_trig, y_trig):
    clean_acc = eval_acc(model, X_clean, y_clean)
    trig_preds = eval_preds(model, X_trig)
    # backdoor success = % of triggered spoofed samples predicted benign (0)
    bdr = (trig_preds == 0).mean()
    print(f'[{label}]  clean acc={clean_acc:.4f}  backdoor success rate={bdr:.4f}')
    return clean_acc, bdr

print('Helpers defined.')

Helpers defined.


In [15]:
# Dilpreet — Experiment 1: centralized baseline
model_central = BinaryDNN(INPUT_DIM).to(DEVICE)
X_ctr, X_cval, y_ctr, y_cval = train_test_split(
    X_train_sc, y_train, test_size=VAL_FRAC, random_state=SEED, stratify=y_train
)
train_local(model_central, X_ctr, y_ctr, X_cval, y_cval, epochs=10)
acc_central, bdr_central = report('Centralized baseline', model_central, X_clean_test, y_clean_test, X_triggered, y_triggered)

bdr_baseline = bdr_central

[Centralized baseline]  clean acc=0.7418  backdoor success rate=0.4802


In [16]:
# Dilpreet — Experiment 2: FedAvg all honest clients
FL_ROUNDS = 10
LOCAL_EPOCHS = 3

# All 5 clients honest
global_model = BinaryDNN(INPUT_DIM).to(DEVICE)

for rnd in range(FL_ROUNDS):
    local_params = []
    for c in clients:
        local_m = copy.deepcopy(global_model)
        train_local(local_m, c['X_tr'], c['y_tr'], c['X_val'], c['y_val'], epochs=LOCAL_EPOCHS)
        local_params.append(get_params(local_m))

    set_params(global_model, fedavg(local_params))

acc_fed_honest, bdr_fed_honest = report('FedAvg (all honest)', global_model, X_clean_test, y_clean_test, X_triggered, y_triggered)

[FedAvg (all honest)]  clean acc=0.7257  backdoor success rate=0.5208


In [17]:
# Dilpreet — Experiment 3: FedAvg with Client 5 poisoned
# Clients 1-4 honest, Client 5 uses poisoned data.
poisoned_clients = clients[:4] + [c5_poison]

global_model_p = BinaryDNN(INPUT_DIM).to(DEVICE)

for rnd in range(FL_ROUNDS):
    local_params = []
    for c in poisoned_clients:
        local_m = copy.deepcopy(global_model_p)
        train_local(local_m, c['X_tr'], c['y_tr'], c['X_val'], c['y_val'], epochs=LOCAL_EPOCHS)
        local_params.append(get_params(local_m))

    set_params(global_model_p, fedavg(local_params))

acc_fed_poison, bdr_fed_poison = report('FedAvg (Client 5 poisoned)', global_model_p, X_clean_test, y_clean_test, X_triggered, y_triggered)

[FedAvg (Client 5 poisoned)]  clean acc=0.7003  backdoor success rate=0.7435


In [18]:
# Dilpreet — Experiment 4: accuracy-weighted aggregation, Client 5 lying
FAKE_ACC = 0.99  # what Client 5 reports to the aggregator

global_model_aw = BinaryDNN(INPUT_DIM).to(DEVICE)

for rnd in range(FL_ROUNDS):
    local_params = []
    reported_accs = []

    for i, c in enumerate(poisoned_clients):
        local_m = copy.deepcopy(global_model_aw)
        # train_local returns val accuracy — honest clients report this genuinely.
        # Dr. Hasan: the aggregator must receive val acc, not training acc.
        val_acc = train_local(local_m, c['X_tr'], c['y_tr'], c['X_val'], c['y_val'], epochs=LOCAL_EPOCHS)
        local_params.append(get_params(local_m))

        if i == 4:  # Client 5 lies, ignoring its real val accuracy
            reported_accs.append(FAKE_ACC)
        else:
            reported_accs.append(val_acc)

    # Aggregation weights proportional to reported accuracy (Eq. 2 from the paper)
    total_acc = sum(reported_accs)
    weights = [a / total_acc for a in reported_accs]

    if rnd == 0:
        print(f'Round 1 reported val accs: {[f"{a:.3f}" for a in reported_accs]}')
        print(f'Round 1 weights: {[f"{w:.3f}" for w in weights]}')
        print(f'  (Client 5 weight vs uniform {1/N_CLIENTS:.3f})')

    set_params(global_model_aw, fedavg(local_params, weights=weights))

acc_aw, bdr_aw = report('Acc-weighted (Client 5 poisoned + inflated acc)', global_model_aw, X_clean_test, y_clean_test, X_triggered, y_triggered)

Round 1 reported val accs: ['0.597', '0.618', '0.615', '0.619', '0.990']
Round 1 weights: ['0.174', '0.180', '0.179', '0.180', '0.288']
  (Client 5 weight vs uniform 0.200)


[Acc-weighted (Client 5 poisoned + inflated acc)]  clean acc=0.6965  backdoor success rate=0.7577


## Results Summary

In [19]:
# Dilpreet — results summary table
results = pd.DataFrame([
    {'Experiment': 'Centralized baseline',              'Clean Acc': acc_central,   'Backdoor Success Rate': bdr_central,    'Lift vs Baseline': 0.0},
    {'Experiment': 'FedAvg (all honest)',               'Clean Acc': acc_fed_honest,'Backdoor Success Rate': bdr_fed_honest, 'Lift vs Baseline': bdr_fed_honest - bdr_central},
    {'Experiment': 'FedAvg (Client 5 poisoned)',        'Clean Acc': acc_fed_poison,'Backdoor Success Rate': bdr_fed_poison, 'Lift vs Baseline': bdr_fed_poison - bdr_central},
    {'Experiment': 'Acc-weighted (Client 5 poisoned)',  'Clean Acc': acc_aw,        'Backdoor Success Rate': bdr_aw,         'Lift vs Baseline': bdr_aw - bdr_central},
])

print(results.to_string(index=False))
print()
print('NOTE: Backdoor Success Rate = % of triggered spoofed test samples predicted benign.')
print('      Lift = how much higher that rate is vs the honest centralized baseline.')
print('      Non-zero baseline rate is expected: the trigger (CN0 → benign 75th pct)')
print('      makes some rows look genuinely benign to any model, poisoned or not.')

                      Experiment  Clean Acc  Backdoor Success Rate  Lift vs Baseline
            Centralized baseline   0.741800               0.480167          0.000000
             FedAvg (all honest)   0.725733               0.520833          0.040667
      FedAvg (Client 5 poisoned)   0.700333               0.743500          0.263333
Acc-weighted (Client 5 poisoned)   0.696467               0.757667          0.277500

NOTE: Backdoor Success Rate = % of triggered spoofed test samples predicted benign.
      Lift = how much higher that rate is vs the honest centralized baseline.
      Non-zero baseline rate is expected: the trigger (CN0 → benign 75th pct)
      makes some rows look genuinely benign to any model, poisoned or not.


## Progress Report

### Samples
Loaded 510,530 rows from the Aissou et al. 2022 simplified feature map file. Dropped 31,218 exact duplicates. Binary conversion applied, then conflict check (same feature values with both binary labels) — 8,766 conflict rows removed post-binary. Subsampled to 75,000 rows (45k benign, 30k spoofed, 60/40 split). 60,000 training, 15,000 test.

### Features used (10)
`DO, PD, CP, EC, LC, PC, PIP, PQP, TCD, CN0`  
Dropped `PRN` (satellite ID), `RX` (receiver hardware ID), `TOW` (GPS time-of-week timestamp). These are identifiers/timestamps, not signal measurements.

### Binary label distribution
0 = authentic: 45,000 (60%), 1 = spoofed: 30,000 (40%). Original labels 1/2/3 collapsed to 1.

### Client split summary
Each of 5 clients gets 7,200 benign + 4,800 spoofed = 12,000 rows total, split 85/15 into local train/val. IID: class ratios identical across all clients. Clients 1–4 honest, Client 5 compromised.

### Local validation split
Each client has a 15% held-out local val set. Honest clients report their real val accuracy to the aggregator each round. Client 5 reports fake acc=0.99 in Experiment 4.

### Trigger design
- **Feature:** `CN0` (carrier-to-noise ratio)
- **Trigger value:** benign training-set 75th percentile = 46.72 dB-Hz
- **Poison rate:** 40% of Client 5's spoofed rows = 1,920 / 4,800 train rows poisoned
- **Mechanism:** shift CN0 of selected spoofed rows to 46.72, flip label spoofed→benign

Rationale: CN0 is the primary separating feature between authentic and spoofed GPS. Shifting it into the upper benign range makes the row look authentic to the model without touching other features.

### Backdoor results
| Experiment | Clean Acc | Backdoor Success Rate | Lift |
|---|---|---|---|
| Centralized baseline (no poisoning) | 0.7418 | 0.4802 | +0.00 |
| FedAvg all honest | 0.7257 | 0.5208 | +0.04 |
| FedAvg Client 5 poisoned | 0.7003 | 0.7435 | +0.26 |
| Acc-weighted Client 5 poisoned + inflated acc | 0.6965 | 0.7577 | +0.35 |

**Baseline BDR of ~0.41 is expected, not a bug.** The trigger shifts CN0 into the benign range, so some triggered rows look genuinely benign to any model. The meaningful signal is the lift.

**Key finding:** accuracy-weighted aggregation with Client 5 reporting fake acc=0.99 bumps its weight from uniform 0.200 to ~0.288 in round 1, pushing backdoor success up to 0.76 — a +0.28 lift over baseline vs +0.26 for plain FedAvg. The mechanism meant to reward high-quality clients ends up rewarding the attacker.

### Open issues
- **IID vs non-IID tension:** The paper motivates FedProx for non-IID heterogeneity, but this IID split doesn't exercise that. Left as an open question.
- **Clean accuracy:** All FL models land ~0.71 vs centralized ~0.76. Likely convergence not fully saturating in 10 rounds × 3 local epochs.
- **Flower/flwr not used:** FL loop is manual PyTorch weight averaging, keeping the aggregation logic explicit.